# 05 — Debugging the Context Attribution Module

End-to-end debug harness for `context_attribution.context_cite`. This mimics a
**full test run** of the pipeline but is wired for inspection: every stage is
broken out so you can see the inputs and outputs as data flows through the module.

**What this notebook does and does not do:**

- **No backend API call.** The RAG response is loaded from the local
  `chat_output.txt` mock (a real captured backend payload), so `fetch_backend`
  is never invoked. This isolates the attribution module from the backend.
- **Still calls the Zingplay vLLM endpoint** for ContextCite. Every ablation
  scores the response with `prompt_logprobs=1` against the live LLM endpoint
  configured in `.env` (`OPENAI_BASE_URL` / `OPENAI_API_KEY`). That call path
  is exactly what we're debugging.

Pipeline mirrors `python -m context_attribution`:
`load mock → prepare_inputs → ContextCiter → get_attributions`.

## Setup — import the module and inspect its config

In [ ]:
import sys, json
from pathlib import Path
from IPython.display import display

sys.path.insert(0, str(Path('..').resolve()))

from dotenv import load_dotenv
load_dotenv(dotenv_path='../.env')

# Import the module under test. It self-configures from .env on import:
# the Zingplay LLM endpoint for prompt_logprobs and the mini-model char limits.
import context_attribution.context_cite as cc_module
from context_attribution.context_cite import (
    prepare_inputs,
    make_tokenizer,
    attribute_response,
    APIModel,
)
from context_cite import ContextCiter
from context_cite.context_citer import DEFAULT_PROMPT_TEMPLATE

print('Module under test   :', cc_module.__file__)
print('LLM endpoint (ZP)   :', cc_module.LLM_URL)
print('Model               :', cc_module.MODEL)
print('LLM API key set     :', bool(cc_module.API_KEY))
print('Max response chars  :', cc_module.MINI_MAX_RESPONSE_CHARS)
print('Max context chars   :', cc_module.MINI_MAX_CONTEXT_CHARS)

# Sanity: we need the Zingplay endpoint for ContextCite, but NOT the backend.
assert cc_module.API_KEY,  'OPENAI_API_KEY missing — ContextCite needs the Zingplay endpoint'
assert cc_module.LLM_URL.startswith('http'), f'LLM_URL looks unset: {cc_module.LLM_URL!r}'
print('\nReady: Zingplay endpoint configured, backend will NOT be called.')

## Stage 1 — Load mock data (no backend call)

`chat_output.txt` is a captured backend payload (the same shape `fetch_backend`
returns). We load it straight from disk and pair it with the query that produced
it, so the backend is bypassed entirely.

In [ ]:
MOCK_PATH = Path('chat_output.txt')
QUERY = 'Nhân viên VNG được nghỉ phép bao nhiêu ngày mỗi năm?'

# Load directly from file — fetch_backend() is deliberately NOT used here.
with open(MOCK_PATH) as f:
    data = json.load(f)

print('Loaded mock payload :', MOCK_PATH.resolve())
print('Query               :', QUERY)
print('Top-level keys      :', list(data.keys()))
print()
print(f'answer              : {len(data["answer"])} chars')
print(f'knowledge_sources   : {len(data["knowledge_sources"])} chunks')
print(f'mode                : {data.get("mode")}')
print(f'usage               : {data.get("usage")}')
print()
print('--- raw answer ---')
print(data['answer'])
print()
print('--- knowledge source chunks ---')
for i, src in enumerate(data['knowledge_sources']):
    snip = ' '.join(src.get('content', '')[:100].split())
    print(f'  [{i}] score={src.get("score", 0):.3f}  {src.get("document_id", "<no doc>")}')
    print(f'       {snip}...')

## Stage 2 — `prepare_inputs`: build context + response

Applies the mini-model capacity limits (`MINI_MAX_RESPONSE_CHARS` /
`MINI_MAX_CONTEXT_CHARS`) and concatenates the surviving chunks into the context
string. Watch the printed `Chunks: a/b` line — if chunks are dropped here, the
attribution can only ever credit the chunks that survived truncation.

In [ ]:
context, response = prepare_inputs(data)

print('=== CONTEXT (', len(context), 'chars ) ===')
print(context)
print()
print('=== RESPONSE (', len(response), 'chars ) ===')
print(response)

if len(response) < len(data['answer']):
    print(f'\n[debug] response was truncated: {len(data["answer"])} -> {len(response)} chars')

## Stage 3 — Wire up `ContextCiter` and inspect the source partition

`make_tokenizer()` returns the GPT-2 shell tokenizer (ChatML template, used only
for structural bookkeeping). `APIModel` holds the fixed response so `generate()`
returns it without a live call. ContextCite then partitions the context into
sentence-level **sources** — these are the candidate attribution units.

`cc.response` triggers `generate()` (no API call). `cc.sources` shows the
partition that the ablations will toggle.

In [ ]:
tokenizer = make_tokenizer()
model = APIModel(response=response)

NUM_ABLATIONS = 32  # full-test setting; lower to ~8 for a faster debug loop

cc = ContextCiter(
    model=model,
    tokenizer=tokenizer,
    context=context,
    query=QUERY,
    num_ablations=NUM_ABLATIONS,
    ablation_keep_prob=0.5,
    batch_size=1,
)

print(f'num_sources : {cc.num_sources}')
print(f'num_ablations: {NUM_ABLATIONS}')
print()
print('Sources (attribution units):')
for i, s in enumerate(cc.sources):
    print(f'  [{i}] {s[:90]}{"..." if len(s) > 90 else ""}')

print()
print('cc.response (via generate(), no API call):')
print(' ', cc.response[:200], '...' if len(cc.response) > 200 else '')

## Stage 4 — Debug a single Zingplay `prompt_logprobs` call

Before running all the ablations, verify the ContextCite scoring path works
against the live Zingplay endpoint with the **full (unmasked)** context. This is
the exact call `APIModel.__call__` makes per ablation; if this fails, the whole
run fails. We reconstruct the user content the way ContextCite does, then call
the module's `_api_response_token_logprobs` helper directly.

In [ ]:
# Reconstruct the unmasked user content exactly as ContextCite builds it.
user_content = DEFAULT_PROMPT_TEMPLATE.format(context=context, query=QUERY)
print('user_content (', len(user_content), 'chars ) head:')
print(user_content[:300], '...\n')

# Live Zingplay call — this is the ContextCite scoring path under test.
api_tokens = cc_module._api_response_token_logprobs(user_content, response)

print(f'Zingplay returned {len(api_tokens)} scored response tokens')
total_lp = sum(lp for _, lp in api_tokens)
print(f'sum logprob = {total_lp:.3f}   mean = {total_lp / max(len(api_tokens), 1):.3f}')
print()
print('First 20 (token, logprob):')
for tok, lp in api_tokens[:20]:
    print(f'  {tok!r:20} {lp:8.3f}')

assert api_tokens, 'Zingplay returned no token logprobs — check endpoint / model / prompt size'
print('\nSingle-call scoring path OK.')

## Stage 5 — Full attribution run (mimics the real test)

This is the full pipeline end-to-end: ContextCite masks random subsets of the
sources across `num_ablations` calls to Zingplay (`prompt_logprobs=1`), then fits
a Lasso to recover which sources drove the response. Equivalent to
`attribute_response(...)` / `python -m context_attribution`.

`verbose=True` prints the per-ablation progress so you can watch the live calls.

In [ ]:
# Runs NUM_ABLATIONS live calls to the Zingplay endpoint.
result = cc.get_attributions(as_dataframe=True, verbose=True)
display(result)

## Stage 6 — Validate the attribution output

Sanity checks on the result a full test would assert: a row per source, finite
weights, and a clearly-ranked top contributor. The top source should be the
policy chunk that actually states the leave-day numbers (16 / 14 days).

In [ ]:
df = result.data if hasattr(result, 'data') else result
print('shape        :', df.shape)
print('columns      :', list(df.columns))
assert df.shape[0] == cc.num_sources, 'row count should equal number of sources'

score_col = 'Score' if 'Score' in df.columns else df.columns[0]
ranked = df.sort_values(score_col, ascending=False)
print()
print('Top sources by attribution score:')
print(ranked.head(5).to_string())

import numpy as np
assert np.isfinite(df[score_col].astype(float)).all(), 'non-finite attribution score'
print('\nAttribution output looks well-formed. Full debug run complete.')